# Iterative Graph


In [4]:
from langgraph.graph import StateGraph , START , END 
from typing import TypedDict, Literal, Annotated
from langchain_core.messages import SystemMessage  , HumanMessage
from langchain_openai import ChatOpenAI

import operator

from dotenv import load_dotenv

load_dotenv()


True

# Seperate models for each tak

## Workflow at a glance
This graph repeatedly improves a generated Facebook post:

1. The generator creates an initial draft from the topic.
2. The evaluator returns a structured decision (`approved` or `needs_improvement`) plus feedback.
3. If improvement is needed, the optimizer revises the post and the evaluator checks it again.
4. The loop ends when the post is approved or `max_iteration` is reached.

The generator, evaluator, and optimizer use separate model instances. `post_history` and `feedback_history` keep the successive drafts and reviews; their `operator.add` reducers append updates instead of replacing earlier entries. Reaching the iteration limit stops the loop even if the final evaluation still requests improvements.


In [ ]:
# Give each stage its own model instance so its role can be configured independently.
generator_llm = ChatOpenAI(model="gpt-4.1-mini")
evaluator_llm = ChatOpenAI(model="gpt-4.1-mini")
optimizer_llm = ChatOpenAI(model="gpt-4.1-mini")


In [ ]:
from pydantic import BaseModel, Field

# Validate the evaluator's response: a decision from the allowed choices plus feedback.
class PostEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation result")
    feedback: str = Field(..., description="Feedback explaining the evaluation")


In [ ]:
# Constrain evaluator output to the fields and decision values in PostEvaluation.
structured_evaluator_llm = evaluator_llm.with_structured_output(PostEvaluation)


In [8]:
post = """🤖 The Next Evolution of AI: Agentic AI 🚀

We have all used AI tools like ChatGPT to ask questions, generate content, or solve problems. But what if AI could do more than just respond to our prompts?

Imagine an AI system that can:

🧠 Understand a goal and break it into smaller tasks.

🔍 Search the web or retrieve information when needed.

🛠️ Use tools, APIs, and databases to perform actions.

🔄 Observe the results, decide what to do next, and adapt its approach.

✅ Work through multiple steps to accomplish a task.

Thats the idea behind Agentic AI!

For example, instead of simply asking an AI to plan a trip, you could have an AI agent research destinations, compare flights, check the weather, and prepare an itinerary using external tools.

The fascinating part is that the AI can decide which action to take next based on the information it receives.

Of course, agents still need guardrails, verification, and human approval for important or sensitive actions. They are powerful, but not magic!

Im currently exploring this space through Python, LLMs, RAG, LangChain, and LangGraph — learning how to combine AI reasoning with backend engineering to build more capable applications.

We are moving from AI that simply answers questions toward AI systems that can help accomplish goals. 🚀

The big question is: What will you build when AI can do more than just generate text?

#AgenticAI #ArtificialIntelligence #GenerativeAI #Python #TechInnovation"""


In [9]:
result = structured_evaluator_llm.invoke(post)

In [10]:
result.evaluation

'approved'

In [11]:
result.feedback

'The post effectively introduces the concept of Agentic AI in an engaging and accessible manner. It clearly outlines the capabilities of such AI systems compared to conventional AI tools, providing a relatable example with trip planning. The tone is informative yet inviting, encouraging readers to think about future applications. The mention of technologies like Python, LLMs, RAG, LangChain, and LangGraph adds credibility and showcases expertise. Including a balanced note on the need for guardrails and human oversight enhances trustworthiness. Overall, the post is well-structured, informative, and likely to resonate well with a tech-savvy audience.'

In [ ]:
# State passed between nodes. Each node returns only the fields it updates.
class PostState(TypedDict):
    topic: str
    post: str
    evaluation: Literal["approved", "needs_improvement"]
    feedback: str
    iteration: int
    max_iteration: int

    # operator.add concatenates each node's list update with the existing history.
    post_history: Annotated[list[str], operator.add]
    feedback_history: Annotated[list[str], operator.add]


In [20]:
def generate_post(state: PostState):

    # Prompt the generator with the topic and constraints for the first draft.
    messages = [
        SystemMessage(content="You are a funny and clever Facebook influencer."),
        HumanMessage(content=f"""
Write a short, original, and hilarious Facebook post on the topic: "{state['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 500 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english
""")
    ]

    # Generate the first draft; the content becomes part of the graph state.
    response = generator_llm.invoke(messages).content

    # Return the post and its first history entry so evaluate_post can read `post`.
    return {'post': response, 'post_history': [response]}


In [25]:
def evaluate_post(state: PostState):
    """Rate the current draft and append the review to feedback history."""
    messages = [
        SystemMessage(content="You are a ruthless, no-laugh-given Facebook critic. You evaluate posts based on humor, originality, virality, and post format."),
        HumanMessage(content=f"""
Evaluate the following Facebook post:

Post: "{state['post']}"

Use the criteria below to evaluate the post:

1. Originality – Is this fresh, or have you seen it a hundred times before?  
2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
3. Punchiness – Is it short, sharp, and scroll-stopping?  
4. Virality Potential – Would people share, react, or comment on it?  
5. Format – Is it a well-formed Facebook post (not a setup-punchline joke, not a Q&A joke, and under 500 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 500 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"  
- feedback: One paragraph explaining the strengths and weaknesses 
""")
    ]

    response = structured_evaluator_llm.invoke(messages)

    # Store the latest decision/feedback and append this review to the history.
    return {'evaluation': response.evaluation, 'feedback': response.feedback, 'feedback_history': [response.feedback]}


In [26]:
def optimize_post(state: PostState):
    """Revise the current post using the evaluator's latest feedback."""
    messages = [
        SystemMessage(content="You punch up Facebook posts for virality and humor based on given feedback."),
        HumanMessage(content=f"""
Improve the Facebook post based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Post:
{state['post']}

Re-write it as a short, viral-worthy Facebook post. Avoid Q&A style and stay under 500 characters.
""")
    ]

    response = optimizer_llm.invoke(messages).content

    # A revision creates the next iteration and is appended to post history.
    iteration = state['iteration'] + 1
    return {'post': response, 'iteration': iteration, 'post_history': [response]}


In [27]:
def route_evaluation(state: PostState):
    """End after approval or the iteration cap; otherwise send the post for revision."""
    # 'approved' routes to END in the graph; the cap stops even if still unapproved.
    if state['evaluation'] == 'approved' or state['iteration'] >= state['max_iteration']:
        return 'approved'
    return 'needs_improvement'


In [28]:
# Build a graph that carries PostState through each generation/evaluation cycle.
graph = StateGraph(PostState)

# Register the draft, review, and revision steps.
graph.add_node('generate', generate_post)
graph.add_node('evaluate', evaluate_post)
graph.add_node('optimize', optimize_post)

# First create a draft, then evaluate it.
graph.add_edge(START, 'generate')
graph.add_edge('generate', 'evaluate')

# The evaluator either ends the workflow or routes the post to optimization.
graph.add_conditional_edges(
    'evaluate',
    route_evaluation,
    {'approved': END, 'needs_improvement': 'optimize'},
)

# After revision, send the updated post back for another evaluation.
graph.add_edge('optimize', 'evaluate')

# Compile the graph so it can be invoked with an initial state.
workflow = graph.compile()


In [ ]:
# Start with a topic, iteration counter, and maximum number of drafts/revisions.
initial_state = {
    "topic": "agentic AI",
    "iteration": 1,
    "max_iteration": 5
}

# Run until approval or the iteration limit, then inspect the final state.
result = workflow.invoke(initial_state)


In [ ]:
# Review the final post, decision, feedback, and accumulated histories.
result


{'topic': 'agentic AI',
 'post': 'Agentic AI: finally a robot boss who actually listens… until it decides your coffee preference is a “critical system inefficiency” and replaces it with 0% caffeine. At least it’s consistent—just like my printer’s desire to sabotage my life. #RobotRebels #CoffeeCrisis2024',
 'evaluation': 'approved',
 'feedback': 'This post stands out with its originality by cleverly combining AI workplace dynamics and everyday frustrations with office printers and coffee preferences. It effectively uses humor that is relatable and witty, especially the personification of AI as a boss with idiosyncratic logic. The post is punchy and sharp, making it scroll-stopping without feeling lengthy or cumbersome. The hashtags add a viral component, inviting engagement through humor and topical tech references. It avoids question-answer formats and setup-punchline structures, and is well within the character limit, making it a well-crafted Facebook post.',
 'iteration': 1,
 'max_i

In [ ]:
# Print each evaluator response accumulated across the iterations.
for feedback_entry in result['feedback_history']:
    print(feedback_entry)


This post stands out with its originality by cleverly combining AI workplace dynamics and everyday frustrations with office printers and coffee preferences. It effectively uses humor that is relatable and witty, especially the personification of AI as a boss with idiosyncratic logic. The post is punchy and sharp, making it scroll-stopping without feeling lengthy or cumbersome. The hashtags add a viral component, inviting engagement through humor and topical tech references. It avoids question-answer formats and setup-punchline structures, and is well within the character limit, making it a well-crafted Facebook post.
